In [1]:
# Install and import LangChain
#!pip install pandas "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4"   # only needs to be done once
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.chat_models import ChatOllama
from pydantic import BaseModel, Field
from typing import List

In [2]:
# Define a prompt template with placeholders
template = """You are an Industrial-Organizational psychologist creating interview questions.

Job Title: {job_title}
Competency: {competency}
Competency Definition: {competency_definition}

Generate 3 behavioral interview questions that assess this competency
for this role.

{format_instructions}
"""

# Define input variables according to the curly brackets in prompt
# Note. format_instructions is set via partial_variables so the parser's
# instructions are automatically injected into every prompt
prompt = PromptTemplate(
    input_variables=["job_title", "competency", "competency_definition"],
    partial_variables={"format_instructions": ""},  # placeholder, will be filled after parser is created
    template=template
)

In [3]:
# Define what structure we want the LLM to return
class InterviewQuestion(BaseModel):
    question: str = Field(description="The interview question text")
    explanation: str = Field(description="Why this question assesses the competency")

class QuestionSet(BaseModel):
    questions: List[InterviewQuestion]

# Create a parser that enforces this structure
parser = PydanticOutputParser(pydantic_object=QuestionSet)

In [4]:
# Initialize the LLM (requires downloading and installing Ollama: https://ollama.com)
#!ollama pull llama3    # Download the model (only needs to be done once)

llm = ChatOllama(model="llama3", temperature=0.7)

# Inject the parser's format instructions into the prompt
prompt_with_instructions = prompt.partial(format_instructions=parser.get_format_instructions())

# Connect all the pieces together
chain = prompt_with_instructions | llm | parser

/var/folders/x3/cy3pk_yd40bfc372nc57wp9m0000gq/T/ipykernel_67999/2104500339.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3", temperature=0.7)


In [ ]:
# Your data: multiple jobs and competencies
jobs_data = [
    {"job_title": "Software Engineer",
     "competency": "Problem Solving",
     "competency_definition": "Ability to analyze complex problems."},
    {"job_title": "Product Manager",
     "competency": "Stakeholder Management",
     "competency_definition": "Ability to manage diverse stakeholder needs."},
    {"job_title": "Data Scientist",
     "competency": "Analytical Thinking",
     "competency_definition": "Ability to interpret complex data patterns."},
    # In practice: 500+ entries from your job analysis database
] # This can also be stored and read from a spreadsheet (e.g., csv file)

# Process all at once in parallel
results = chain.batch(jobs_data, config={"max_concurrency": 3})

# Export to spreadsheet
import pandas as pd
df = pd.DataFrame([
    {"job": d["job_title"], "competency": d["competency"],
     "question": q.question, "explanation": q.explanation}
    for d, r in zip(jobs_data, results)
    for q in r.questions
])
df.to_csv("interview_questions.csv", index=False)
